# The following is the new code:

NOTE: need a CUDA GPU for bnb. CPU is not compatible with this.

Versions used:

    bitsandbytes version:  0.48.1
    torch version:  2.8.0+cu126

Installs:

In [72]:
!pip install bitsandbytes
!pip install torch
!pip install tqdm

Imports:

In [64]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import bitsandbytes as bnb
from torch.utils.data import Dataset, DataLoader
import json
from tqdm import tqdm

In [71]:
print(f'bitsandbytes version: ', bnb.__version__)
print(f'torch version: ', torch.__version__)

bitsandbytes version:  0.48.1
torch version:  2.8.0+cu126


Dataset:

Defines our dataset for future use.

In [65]:
class ChessDataset(Dataset):
    def __init__(self, tensor_data):
        self.data = tensor_data

    def __len__(self):
        return self.data.size(0)

    def __getitem__(self, idx):
        x = self.data[idx, :-1]
        y = self.data[idx, 1:]
        return x, y

Sparse Multihead Attention:

This is a sparse self-attention layer that limits how far each token can attend in the past using a sliding window (sparsity window).
This makes computation more efficient :D

In [66]:
class SparseMultiheadAttention(nn.Module):
    def __init__(self, d_model, nhead, sparsity_window=32):
        super().__init__()
        self.nhead = nhead
        self.sparsity_window = sparsity_window
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)

    def forward(self, x):
        T = x.size(1)

        # Sparse mask: prevent attending to tokens beyond a sliding window

        mask = torch.ones((T, T), device=x.device).triu(1)

        if self.sparsity_window < T:
            mask[:, :-self.sparsity_window] = 1

        mask = mask.bool()
        out, _ = self.attn(x, x, x, attn_mask=mask)
        return out

Sparse Decode Layer:

A layer of a transformer decoder block involving sparse attn.

In [67]:
class SparseDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, sparsity_window=32, dropout=0.1):
        super().__init__()
        self.self_attn = SparseMultiheadAttention(d_model, nhead, sparsity_window)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_out = self.self_attn(x)
        x = x + self.dropout(attn_out)
        x = self.norm1(x)

        ff_out = self.linear2(F.relu(self.linear1(x)))
        x = x + self.dropout(ff_out)
        x = self.norm2(x)

        return x

Decoder:

Overall the decoder architecture (just with sparse attn).

In [68]:
class ExpandedAttentionChessDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, max_len, sparsity_window=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = nn.ModuleList([
            SparseDecoderLayer(d_model, nhead, sparsity_window=sparsity_window)
            for _ in range(num_layers)
        ])

        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.size()
        positions = torch.arange(0, T, device=x.device).unsqueeze(0)
        x = self.embed(x) + self.pos_emb(positions)

        for layer in self.layers:
            x = layer(x)

        return self.fc_out(x)

Quantized Model:

This just replaces the nn.Linear layers with 8-bit versions from bnb.
Makes things more efficient :D

In [69]:
def quantize_model_8bit(model):
    for name, module in model.named_children():

        if isinstance(module, nn.Linear):
            setattr(model, name, bnb.nn.Linear8bitLt(
                module.in_features,
                module.out_features,
                bias=module.bias is not None
            ))

        else:
            quantize_model_8bit(module)

    return model

# Model Training

Note: Set batch size, sequence length, num layers, d_model, epochs to initial settings.


Also, remove "encoded_tensor = encoded_tensor[:, :256]  # FOR TESTING --- REMOVE LATER!!!". This was for testing. It speeds it up a lot...

You may also need to reset the dataset being used depending on what you want to use.


BTW this is where the training is (shocker).
This is what trains our newly quantized, sparse attention decoder transformer on the training dataset.

In [70]:
if __name__ == "__main__":
    encoded_tensor = torch.load("encoded_games_small.pt")

    encoded_tensor = encoded_tensor[:, :256]  # FOR TESTING --- REMOVE LATER!!!

    with open("move_to_id.json", "r") as f:
        move_to_id = json.load(f)

    dataset = ChessDataset(encoded_tensor)
    loader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)

    vocab_size = len(move_to_id)
    PAD_ID = move_to_id["<PAD>"]

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on device: {device}")

    with open("best_hparams.json", "r") as f:
        best_hparams = json.load(f)

    params = best_hparams["best_params"]

    # Build model on CPU first
    model = ExpandedAttentionChessDecoder(
        vocab_size=vocab_size,
        # d_model=params["d_model"],  # Commented out for testing
        d_model=128,
        nhead=params["nhead"],
        # num_layers=params["num_layers"],  # Commented out for testing
        num_layers=2,
        max_len=encoded_tensor.size(1),
        sparsity_window=32  # can adjust to 32 for even higher sparsity
    )

    # Quantize while still on CPU
    model = quantize_model_8bit(model)

    # Now move to GPU
    model = model.to(device)
    model.train()
    print("Model quantized to 8-bit and moved to CUDA with sparse attention.")

    optimizer = bnb.optim.Adam8bit(model.parameters(), lr=params["lr"])
    print("Using bnb Adam8bit optimizer.")

    # Training!

    for epoch in range(1):
      epoch_loss = 0
      num_batches = 0

      progress_bar = tqdm(loader, desc=f"Epoch {epoch+1}", leave=True)
      for x, y in progress_bar:
          x, y = x.to(device), y.to(device)

          logits = model(x)
          loss = F.cross_entropy(
              logits.reshape(-1, vocab_size),
              y.reshape(-1),
              ignore_index=PAD_ID
          )

          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

          epoch_loss += loss.item()
          num_batches += 1
          avg_loss = epoch_loss / num_batches

          progress_bar.set_postfix(loss=f"{avg_loss:.4f}")

      print(f"Epoch {epoch+1}: avg loss = {avg_loss:.4f}")

      torch.save(model.state_dict(), f"quant_8bit_epoch{epoch+1}.pt")

      print("Saved quantized model checkpoint.")

Running on device: cuda
Model quantized to 8-bit and moved to CUDA with sparse attention.
Using bnb Adam8bit optimizer.


Epoch 1: 100%|██████████| 778/778 [02:00<00:00,  6.48it/s, loss=8.8500]


Epoch 1: avg loss = 8.8500
Saved quantized model checkpoint.
